# 问题 5：革命检测

识别音乐演化中的「革命」（重大跳跃），以及代表革命的艺术家。

**方法：**
1. 用 `data_by_year` 构造每年的标准化特征向量
2. 计算相邻年份的特征向量距离（变化幅度），滑动平滑后找显著跳变点
3. 把跳变点对照社会/技术事件
4. 革命者 = 高影响（出度）+ 在革命期前后活跃的艺术家

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100

by_year = pd.read_csv('output/by_year.csv')

FEATURES = ['danceability', 'energy', 'valence', 'tempo', 'loudness',
            'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'duration_ms']

Z = (by_year[FEATURES] - by_year[FEATURES].mean()) / by_year[FEATURES].std()
z = Z.values
years = by_year['year'].values

print('年份范围:', years.min(), '-', years.max(), '| 行数:', len(years))

## 1. 变点检测：相邻年份特征向量距离

In [ ]:
# 相邻年份特征向量欧氏距离（变化幅度）
d = np.zeros(len(years))
for t in range(1, len(years)):
    d[t] = np.linalg.norm(z[t] - z[t - 1])

# 滑动平滑
window = 5
d_smooth = pd.Series(d).rolling(window, center=True, min_periods=1).mean().values

threshold = d_smooth.mean() + 2 * d_smooth.std()
change_points = years[d_smooth > threshold]

print(f'阈值 = {threshold:.3f}')
print(f'显著变化年份: {change_points.tolist()}')

plt.figure(figsize=(11, 5))
plt.plot(years, d_smooth, color='#4C72B0', label='smoothed change magnitude')
plt.axhline(threshold, color='#C44E52', ls='--', label=f'threshold (mean+2σ)')
plt.scatter(change_points, d_smooth[np.isin(years, change_points)],
            color='#C44E52', zorder=5, label='change points')
plt.xlabel('Year'); plt.ylabel('Feature-vector change')
plt.title('Music evolution change-point detection')
plt.legend(); plt.tight_layout(); plt.show()

## 2. 关键特征时间序列（标注革命期）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, ['energy', 'valence', 'acousticness']):
    ax.plot(years, by_year[feat], color='#333333')
    for cp in change_points:
        ax.axvline(cp, color='#C44E52', alpha=0.5, lw=0.8)
    ax.set_xlabel('Year'); ax.set_ylabel(feat)
    ax.set_title(f'{feat} over time')
plt.tight_layout(); plt.show()

## 3. 革命者识别

取最大跳变所在年代作为革命期，找出「高影响力 + 该时期活跃」的艺术家。

In [ ]:
edges = pd.read_csv('output/edges.csv', dtype=str)
artists = pd.read_csv('output/artists.csv', dtype=str)

outdeg = edges.groupby('influencer_id').size()

# 最大跳变年份 → 革命期 decade
peak_year = change_points[np.argmax(d_smooth[np.isin(years, change_points)])]
rev_decade = int(peak_year // 10 * 10)
print(f'革命期年代: {rev_decade}s（峰值年 {peak_year}）')

cand = artists.copy()
cand['active_start'] = pd.to_numeric(cand['active_start'], errors='coerce')
cand['outdeg'] = cand['artist_id'].map(outdeg).fillna(0).astype(int)

rev_cand = cand[(cand['outdeg'] > 0) &
                (cand['active_start'] >= rev_decade - 20) &
                (cand['active_start'] <= rev_decade + 9)]

top = rev_cand.sort_values('outdeg', ascending=False).head(12)
print('\n=== 革命期高影响力艺术家 ===')
print(top[['artist_name', 'genre', 'active_start', 'outdeg']].to_string(index=False))

## 4. 结论与历史对照

- 跳变点对应音乐特征发生「超过通常波动」的快速变化的年份，可视为革命的候选信号。
- 常见的历史对照（可在报告中引用文献讨论）：
  - 1950–60s：摇滚兴起、电子音乐实验室建立（Columbia–Princeton），acousticness 下降
  - 二战/金融危机前后：valence 下降
  - 2000s 后：duration 骤降（流媒体/TikTok 影响）
- 注意：数据只显示「何时变化」，社会/技术解释需引用外部文献，不能从时间对应直接断言因果。
- 革命者需满足「高影响力 + 早期作品新颖 + 后续传播」，出度只是初步筛选。